# Phase 2.3: PubMed Abstract Acquisition

This notebook reads Phase 2 search results, skips PMIDs already in the raw article cache, fetches a bounded batch when enabled, and rebuilds the Phase 2 tables from cached records. Each successful MEDLINE record is appended immediately, so an interrupted run can resume.


In [5]:
from pathlib import Path
import sys

from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v2").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v2.abstracts.abstract_exports import (
    _read_jsonl_resume_records,
    _output_paths,
    build_pubmed_client,
    prepare_phase2_search_results,
    run_pubmed_fetches,
    selected_pmids_from_search_results,
    write_phase2_outputs,
)
from pathogen_atlas_v2.abstracts.pubmed_client import load_ncbi_entrez_config
from pathogen_atlas_v2.abstracts.pubmed_query_plan import load_pubmed_query_plan
from pathogen_atlas_v2.abstracts.pubmed_query_templates import load_pubmed_query_template_config
from pathogen_atlas_v2.common.io import read_jsonl, resolve_path
from pathogen_atlas_v2 import notebook_utils as nbutils


In [6]:
NOTEBOOK_CONFIG_PATH = "configs/notebook_parameters.yaml"
PARAMS = nbutils.notebook_phase_parameters("phase2_abstract_fetch", NOTEBOOK_CONFIG_PATH)

TEMPLATE_CONFIG_PATH = PARAMS["template_config_path"]
FILTER_CONFIG_PATH = PARAMS["filter_config_path"]
FORCE_REFRESH = bool(PARAMS["force_refresh"])
RUN_ABSTRACT_FETCH = bool(PARAMS["run_abstract_fetch"])
LOG_EVERY = int(PARAMS["log_every"])

PHASE2_CONFIG = load_pubmed_query_template_config(TEMPLATE_CONFIG_PATH)
PHASE2_PATHS = _output_paths(PHASE2_CONFIG)
QUERY_PLAN_PATH = resolve_path(PHASE2_CONFIG["query_plan"]["path"])
SEARCH_RESULTS_PATH = PHASE2_PATHS["raw_search_results_path"]
ARTICLE_CACHE_PATH = PHASE2_PATHS["raw_article_records_path"]
MAX_PMIDS_THIS_RUN = PHASE2_CONFIG["source"].get("max_pmids_per_fetch_run", 25)
FETCH_BATCH_SIZE = int(PHASE2_CONFIG["source"].get("fetch_batch_size", 200))


In [7]:
# Preflight: count selected IDs, cached records, remaining work, and request estimates.
if not SEARCH_RESULTS_PATH.exists():
    raise FileNotFoundError(f"Missing {SEARCH_RESULTS_PATH}. Run notebook 02.2 first.")
if not QUERY_PLAN_PATH.exists():
    raise FileNotFoundError(f"Missing {QUERY_PLAN_PATH}. Run notebook 02.1 first.")

query_plan = load_pubmed_query_plan(QUERY_PLAN_PATH)
search_results = _read_jsonl_resume_records(SEARCH_RESULTS_PATH)
search_results, _, _ = prepare_phase2_search_results(query_plan, search_results, TEMPLATE_CONFIG_PATH)
selected_pmids = selected_pmids_from_search_results(search_results)
article_records = _read_jsonl_resume_records(ARTICLE_CACHE_PATH) if ARTICLE_CACHE_PATH.exists() and not FORCE_REFRESH else []
cached_pmids = {str(record.get("PMID")) for record in article_records if record.get("PMID")}
pending_pmids = [pmid for pmid in selected_pmids if pmid not in cached_pmids]
scheduled_count = len(pending_pmids) if MAX_PMIDS_THIS_RUN is None else min(len(pending_pmids), int(MAX_PMIDS_THIS_RUN))
request_count = (scheduled_count + FETCH_BATCH_SIZE - 1) // FETCH_BATCH_SIZE
ncbi_config = load_ncbi_entrez_config(PHASE2_CONFIG["source"]["ncbi_config_path"])
rate_key = "rate_limit_with_api_key_per_second" if ncbi_config.api_key else "rate_limit_without_api_key_per_second"
configured_rate = float(PHASE2_CONFIG["source"][rate_key])
request_delay = PHASE2_CONFIG["source"].get("request_delay_seconds")
effective_rate = min(configured_rate, (1 / float(request_delay)) if request_delay else configured_rate)

fetch_preflight = {
    "selected_pmids": len(selected_pmids),
    "cached_article_records": len(cached_pmids),
    "pending_pmids": len(pending_pmids),
    "pmids_in_this_run_if_enabled": scheduled_count,
    "estimated_efetch_requests": request_count,
    "ncbi_api_key_configured": bool(ncbi_config.api_key),
    "effective_rate_limit_per_second": effective_rate,
    "estimated_minimum_request_time_seconds": round(request_count / effective_rate, 1),
    "run_abstract_fetch": RUN_ABSTRACT_FETCH,
    "force_refresh": FORCE_REFRESH,
}
display(fetch_preflight)


{'selected_pmids': 110216,
 'cached_article_records': 17225,
 'pending_pmids': 92991,
 'pmids_in_this_run_if_enabled': 92991,
 'estimated_efetch_requests': 465,
 'ncbi_api_key_configured': True,
 'effective_rate_limit_per_second': 10.0,
 'estimated_minimum_request_time_seconds': 46.5,
 'run_abstract_fetch': True,
 'force_refresh': False}

In [8]:
if RUN_ABSTRACT_FETCH:
    if FORCE_REFRESH and ARTICLE_CACHE_PATH.exists():
        ARTICLE_CACHE_PATH.unlink()
        article_records = []

    # The client uses the email and optional NCBI API key, with a rate-aware delay.
    pubmed_client = build_pubmed_client(PHASE2_CONFIG)
    max_pmids = int(MAX_PMIDS_THIS_RUN) if MAX_PMIDS_THIS_RUN is not None else None
    article_records = run_pubmed_fetches(
        search_results=search_results,
        client=pubmed_client,
        existing_records=article_records,
        output_path=ARTICLE_CACHE_PATH,
        batch_size=FETCH_BATCH_SIZE,
        max_pmids=max_pmids,
        log=True,
        log_every=LOG_EVERY,
    )
    # All fetched records, including abstracts later judged unhelpful, remain cached.
    abstract_result = write_phase2_outputs(
        query_plan,
        search_results,
        article_records,
        TEMPLATE_CONFIG_PATH,
        FILTER_CONFIG_PATH,
        raw_article_records_already_appended=True,
        raw_search_results_already_saved=True,
    )
else:
    abstract_result = {"status": "not_run", "reason": "Set run_abstract_fetch: true in configs/notebook_parameters.yaml to fetch the next batch."}

abstract_summary = nbutils.phase_run_summary(abstract_result)
display(abstract_summary["overview"])
display(abstract_summary["counts"])
display(nbutils.metadata_outputs(abstract_result))


Abstract fetch: 17225/110216 PMIDs already fetched; 92991 remaining; 92991 scheduled this run.
Abstract fetch: 17425/110216 PMIDs fetched; 92791 remaining; 200/92991 attempted this run. ETA 28m 19s (about 2026-10-29 23:48:14).
Abstract fetch: 27225/110216 PMIDs fetched; 82991 remaining; 10000/92991 attempted this run. ETA 18m 13s (about 2026-10-29 23:40:16).
Abstract fetch: 37225/110216 PMIDs fetched; 72991 remaining; 20000/92991 attempted this run. ETA 17m 47s (about 2026-10-29 23:42:31).
Abstract fetch: 47225/110216 PMIDs fetched; 62991 remaining; 30000/92991 attempted this run. ETA 21m 49s (about 2026-10-29 23:52:04).
Abstract fetch: 57224/110216 PMIDs fetched; 52992 remaining; 40000/92991 attempted this run. ETA 21m 49s (about 2026-10-29 23:58:08).
Abstract fetch: 67223/110216 PMIDs fetched; 42993 remaining; 50000/92991 attempted this run. ETA 19m 08s (about 2026-10-30 00:01:14).
Abstract fetch: 77223/110216 PMIDs fetched; 32993 remaining; 60000/92991 attempted this run. ETA 14m 54

,field,value
0,source,pubmed
1,query_plan_path,/Users/f.costa/Documents/Projects/Repositories...
2,query_plan_approval,draft
3,query_count,68397


,section,metric,value
0,record_counts,search_result_rows,68397
1,record_counts,unique_pmids,655584
2,record_counts,sampled_unique_pmids,110216
3,record_counts,raw_article_records,110213
4,record_counts,canonical_species_with_pmids,22799
5,record_counts,release_abstracts,110213
6,record_counts,species_match_rows,2076387
7,record_counts,filter_match_rows,726555


,name,path,exists,rows,size_bytes,content_type
0,raw_search_results_path,/Users/f.costa/Documents/Projects/Repositories...,True,68397,35815970,jsonl
1,canonical_pmid_index_path,/Users/f.costa/Documents/Projects/Repositories...,True,22799,11231861,parquet
2,sampled_canonical_pmid_index_path,/Users/f.costa/Documents/Projects/Repositories...,True,22799,3103426,parquet
3,raw_article_records_path,/Users/f.costa/Documents/Projects/Repositories...,True,110213,426549125,jsonl
4,filtered_species_matches_path,/Users/f.costa/Documents/Projects/Repositories...,True,2076387,484321901,parquet
5,filtered_filter_matches_path,/Users/f.costa/Documents/Projects/Repositories...,True,726555,2492793,parquet
6,release_abstracts_path,/Users/f.costa/Documents/Projects/Repositories...,True,110213,113692555,parquet
7,release_species_matches_path,/Users/f.costa/Documents/Projects/Repositories...,True,2076387,484321901,parquet
8,release_filter_matches_path,/Users/f.costa/Documents/Projects/Repositories...,True,726555,2492793,parquet
9,metadata_path,/Users/f.costa/Documents/Projects/Repositories...,True,1,3342,json


In [9]:
display(nbutils.read_table_if_exists(PHASE2_PATHS["release_abstracts_path"], n_rows=10))


,pmid,title,abstract,publication_date,publication_year,authors,journal,doi,mesh_terms,keywords,source
0,100003,Group G streptococcal endocarditis.,The group G streptococcus may be a more common...,1978 Jul,1978,"[Bouza E, Meyer RD, Busch DF]",American journal of clinical pathology,10.1093/ajcp/70.1.108,"[Diagnosis, Differential, *Endocarditis, Bacte...",[],pubmed
1,100036,[Induction of interferon by Bacillus subtilis ...,Although it is generally admitted that Gram-po...,1978 May-Jun,1978,[Nozaki-Renard J],Annales de microbiologie,,"[Animals, *Bacillus subtilis/analysis, Bacteri...",[],pubmed
2,1000385,Bovine mastitis in Ontario due to Mycoplasma a...,Bovine mastitis caused by Mycoplasma agalactia...,1976 Apr,1976,"[Ruhnke HL, Thawley D, Nelson FC]",Canadian journal of comparative medicine : Rev...,,"[Animals, Cattle, Mastitis, Bovine/blood/*epid...",[],pubmed
3,100162,An apparent outbreak of rickettsial illness in...,A focal outbreak of highly fatal disease occur...,1978,1978,"[Campbell CC, Hobbs JH, Marranghello L, Vargas...",Bulletin of the Pan American Health Organization,,"[Adolescent, Antibodies/analysis, Antigens/ana...",[],pubmed
4,1001829,[Failure of anti-symptomatic vaccination and t...,Blackleg (Clostridium chauvoei) infection in C...,1976,1976,"[Joubert L, Valette L]",Developments in biological standardization,,"[Animals, Bacterial Vaccines/administration & ...",[],pubmed
5,10022069,Technological aspects of the microbial treatme...,Thiobacillus denitrificans has been shown to b...,1998,1998,"[Sublette KL, Kolhatkar R, Raterman K]",Biodegradation,10.1023/a:1008262522493,"[Biodegradation, Environmental, Biomass, Oxida...",[],pubmed
6,1002285,Chronic Clostridium septicum infection of a ti...,An open transverse fracture of the mid-shaft o...,1976 Nov,1976,"[Lucas HK, Speller DC, Stephens M]",Injury,10.1016/0020-1383(76)90044-9,"[Adult, Anti-Bacterial Agents/therapeutic use,...",[],pubmed
7,10024429,Detection of Mycoplasma mycoides subspecies my...,A colorimetric polymerase chain reaction (PCR)...,1999 Feb,1999,"[Bashiruddin JB, De Santis P, Vacciana A, Sant...",Molecular and cellular probes,10.1006/mcpr.1998.0206,"[Animals, Bacteriological Techniques, Buffaloe...",[],pubmed
8,10024542,Role of antibodies against Bordetella pertussi...,Immunization with whole-cell pertussis vaccine...,1999 Mar,1999,"[van den Berg BM, Beekhuizen H, Mooi FR, van F...",Infection and immunity,10.1128/IAI.67.3.1050-1055.1999,"[Adhesins, Bacterial/immunology, Animals, Anti...",[],pubmed
9,10024565,Identification of a cytolethal distending toxi...,A genetic locus for a cytolethal distending to...,1999 Mar,1999,"[Mayer MP, Bueno LC, Hansen EJ, DiRienzo JM]",Infection and immunity,10.1128/IAI.67.3.1227-1237.1999,[Aggregatibacter actinomycetemcomitans/genetic...,[],pubmed


In [10]:
display(nbutils.read_table_if_exists(PHASE2_PATHS["release_species_matches_path"], n_rows=10))


,pmid,canonical_species,matched_names,mapping_types,synonym_count,query_id,query_hash,template_id,field,query,selected_for_fetch
0,100003,Enterococcus faecalis,"[Enterococcus faecalis, Streptococcus faecalis]","[accepted, synonym]",2,q_f58af3fabb8eae1b,f58af3fabb8eae1b,exact_species_human_infection_context,MeSH Terms,"(""Enterococcus faecalis""[MeSH Terms] OR ""Strep...",True
1,1000323,Sphingomonas japonica,"[Parastakelama japonica, Sphingomonas japonica]","[accepted, synonym]",2,q_1940eaab43285177,1940eaab43285177,exact_species_human_infection_context,Keywords,"(""Parastakelama japonica""[Keywords] OR ""Sphing...",False
2,100036,Bacillus subtilis,[Bacillus subtilis],[accepted],1,q_bf27be4ae498f133,bf27be4ae498f133,exact_species_human_infection_context,Keywords,"(""Bacillus subtilis""[Keywords]) AND (""case""[Ke...",True
3,1000385,Mycoplasmopsis agalactiae,"[Mycoplasma agalactiae, Mycoplasmopsis agalact...","[accepted, synonym]",2,q_36feea91fc5fd021,36feea91fc5fd021,exact_species_human_infection_context,Keywords,"(""Mycoplasma agalactiae""[Keywords] OR ""Mycopla...",True
4,1000385,Mycoplasmopsis agalactiae,"[Mycoplasma agalactiae, Mycoplasmopsis agalact...","[accepted, synonym]",2,q_5ee84edb294f3a9f,5ee84edb294f3a9f,exact_species_human_infection_context,Title/Abstract,"(""Mycoplasma agalactiae""[Title/Abstract] OR ""M...",True
5,1000457,Bacteroides fragilis,"[Bacteroides fragilis, Bacteroides hominis]","[accepted, synonym]",2,q_28ca8b27661e4516,28ca8b27661e4516,exact_species_human_infection_context,Keywords,"(""Bacteroides fragilis""[Keywords] OR ""Bacteroi...",False
6,1000457,Bacteroides fragilis,"[Bacteroides fragilis, Bacteroides hominis]","[accepted, synonym]",2,q_87edf6a5902eb40d,87edf6a5902eb40d,exact_species_human_infection_context,Title/Abstract,"(""Bacteroides fragilis""[Title/Abstract] OR ""Ba...",False
7,100048,Neisseria meningitidis,[Neisseria meningitidis],[accepted],1,q_83c05a0e385a29f9,83c05a0e385a29f9,exact_species_human_infection_context,Keywords,"(""Neisseria meningitidis""[Keywords]) AND (""cas...",False
8,100048,Neisseria meningitidis,[Neisseria meningitidis],[accepted],1,q_f470ef7e9699ffa7,f470ef7e9699ffa7,exact_species_human_infection_context,MeSH Terms,"(""Neisseria meningitidis""[MeSH Terms]) AND (""c...",False
9,100049,Neisseria gonorrhoeae,[Neisseria gonorrhoeae],[accepted],1,q_f07c6ad4cb19ec14,f07c6ad4cb19ec14,exact_species_human_infection_context,Keywords,"(""Neisseria gonorrhoeae""[Keywords]) AND (""case...",False
